In [1]:
# ================================================================
# 0) IMPORT & SEED
# ================================================================

import math, random, time, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import mne
from mne.datasets import eegbci
import matplotlib.pyplot as plt
from datetime import datetime

from bci_dataset import BCIIVDataset


mne.set_log_level("ERROR")
# ---------- reproducibility ----------
RANDOM_STATE = 60
def set_seeds(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seeds()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in uso:", DEVICE)

Device in uso: cpu


/home/zino/Projects/EEG/MI/EEG-BCI-Cross-Subject-Motor-Imagery/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [16]:
# ================================================================
# TRACKING
# ================================================================
from dataclasses import dataclass, field

@dataclass
class SubjectHistory:
    subject_id : int
    train_loss : list = field(default_factory=list)
    train_acc  : list = field(default_factory=list)
    val_loss   : list = field(default_factory=list)
    val_acc    : list = field(default_factory=list)
    best_acc   : float = 0.0

    def update(self, tl, ta, vl, va):
        self.train_loss.append(tl)
        self.train_acc.append(ta)
        self.val_loss.append(vl)
        self.val_acc.append(va)
        self.best_acc = max(self.best_acc, va)

    def plot(self, ax_loss, ax_acc):
        ep = range(1, len(self.train_loss) + 1)
        ax_loss.plot(ep, self.train_loss, label="train")
        ax_loss.plot(ep, self.val_loss,   label="val")
        ax_loss.set_title(f"S{self.subject_id:02d} — Loss")
        ax_loss.set_xlabel("Epoca"); ax_loss.legend()

        ax_acc.plot(ep, self.train_acc, label="train")
        ax_acc.plot(ep, self.val_acc,   label="val")
        ax_acc.set_title(f"S{self.subject_id:02d} — Accuracy")
        ax_acc.set_xlabel("Epoca"); ax_acc.legend()


def plot_all_subjects(histories: list[SubjectHistory]):
    """Griglia: una riga per soggetto, colonne Loss | Accuracy."""
    n = len(histories)
    fig, axes = plt.subplots(n, 2, figsize=(12, 3 * n))
    if n == 1: axes = [axes]   # edge case soggetto singolo

    for hist, (ax_l, ax_a) in zip(histories, axes):
        hist.plot(ax_l, ax_a)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()


def plot_final_summary(histories: list[SubjectHistory]):
    """Barchart accuracy per soggetto + media."""
    subjects = [f"S{h.subject_id:02d}" for h in histories]
    accs     = [h.best_acc for h in histories]
    mean_acc = np.mean(accs)

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(subjects, accs, color="steelblue", alpha=0.8)
    ax.axhline(mean_acc, color="red", linestyle="--",
               label=f"Media {mean_acc:.3f}")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Best Val Accuracy")
    ax.set_title("BCI IV 2a — Accuracy per soggetto")
    ax.legend()
    plt.tight_layout()
    plt.savefig("subject_summary.png", dpi=150)
    plt.show()

In [ ]:
# ================================================================
# COSTANTI BCI IV 2a
# ================================================================
N_CHANNELS_BCICIV   = 22
SAMPLING_FREQ_BCICIV = 250
WINDOW_SEC          = 0.5
WIN_SAMPLES         = int(WINDOW_SEC * SAMPLING_FREQ_BCICIV)  # 125
N_CLASSES_BCICIV    = 4
LABELS_BCICIV       = ["LEFT", "RIGHT", "FOOT", "TONGUE"]
DATA_DIR            = "/home/zino/Data/EEG/BCICIV_2a_gdf"

# Architettura
N_BRANCHES           = 4
DEPTH_PER_BRANCH     = 2
START_KERNEL_SIZE    = 15
KERNEL_INCREMENT     = 2
LSTM_HIDDEN_SIZE     = 768
CLASSIFIER_HIDDEN_DIM= 384

# Training
MAX_EPOCHS   = 50
PATIENCE     = 50
BATCH_SIZE   = 16
LEARNING_RATE= 2.89e-5
WEIGHT_DECAY = 5.82e-4

# Pre-processing
L_FREQ, H_FREQ  = 4.0, 40.0
NOTCH_FREQS     = (60,)
APPLY_ICA       = False   # ⇦ attiva se vuoi ICA


# ================================================================
# MODELLO — parametrizzato su n_channels e win_samples
# ================================================================
class ConvBlock(nn.Module):
    def __init__(self, k, n_channels):
        num_groups = next(g for g in [8, 4, 2, 1] if n_channels % g == 0)
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(n_channels, n_channels, k, padding="same"),
            nn.GroupNorm(num_groups, n_channels), nn.ReLU(),
            nn.Conv1d(n_channels, n_channels, k, padding="same"),
            nn.GroupNorm(num_groups, n_channels), nn.ReLU(),
            nn.MaxPool1d(2),
        )
    def forward(self, x): return self.block(x)


class MultiBranchCNNLSTM(nn.Module):
    def __init__(self, n_classes, win_samples, n_channels):
        super().__init__()
        self.branches, self.lstms = nn.ModuleList(), nn.ModuleList()
        for b in range(N_BRANCHES):
            k   = START_KERNEL_SIZE + b * KERNEL_INCREMENT
            cnn = nn.Sequential(
                *[ConvBlock(k, n_channels) for _ in range(DEPTH_PER_BRANCH)]
            )
            self.branches.append(cnn)
            # calcola flat size con un forward pass fittizio
            with torch.no_grad():
                flat = cnn(torch.randn(1, n_channels, win_samples)).numel()
            self.lstms.append(nn.LSTM(flat, LSTM_HIDDEN_SIZE, batch_first=True))

        self.classifier = nn.Sequential(
            nn.Linear(N_BRANCHES * LSTM_HIDDEN_SIZE, CLASSIFIER_HIDDEN_DIM),
            nn.BatchNorm1d(CLASSIFIER_HIDDEN_DIM), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(CLASSIFIER_HIDDEN_DIM, n_classes),
        )

    def forward(self, x):                        # x: (B, T, C, L)
        B, T, C, L = x.shape
        branch_seq = []
        for cnn, lstm in zip(self.branches, self.lstms):
            feat = cnn(x.reshape(B * T, C, L)).reshape(B, T, -1)
            hseq, _ = lstm(feat)                 # (B, T, H)
            branch_seq.append(hseq)
        h = torch.cat(branch_seq, dim=2)         # (B, T, 4*H)
        return self.classifier(h.reshape(B * T, -1)).view(B, T, -1)


# ================================================================
# TRAIN / EVAL — accuracy corretta sulla macro-window
# ================================================================
def run_epoch(model, loader, optim_, criterion, train=True):
    model.train() if train else model.eval()
    loss_sum, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)   # xb:(B,T,C,L)  yb:(B,)
            if train:
                optim_.zero_grad()

            out = model(xb)                           # (B, T, n_cls)
            B, T = out.shape[:2]

            # loss: ogni sotto-finestra predice la stessa label
            loss = criterion(
                out.reshape(B * T, -1),
                yb.unsqueeze(1).expand(B, T).reshape(B * T)
            )
            if train:
                loss.backward()
                optim_.step()

            # accuracy: voto di maggioranza sulle T sotto-finestre
            preds_macro = out.argmax(-1)              # (B, T)
            voted = preds_macro.mode(dim=1).values    # (B,) moda per macro-window
            correct  += (voted == yb).sum().item()
            total    += B
            loss_sum += loss.item() * (B * T)

    return loss_sum / (total * T), correct / total    # loss media, accuracy macro


# ================================================================
# EXPERIMENT — loop per soggetto
# ================================================================
class Experiment:
    def __init__(self, name, window_sec=WINDOW_SEC):
        self.name    = name
        self.win_sec = float(window_sec)
        self.results = []

    def run(self):
        histories = []

        print("\n" + "=" * 60)
        print(f" {self.name} | WIN={self.win_sec}s | BCI IV 2a")
        print("=" * 60)

        for subject_id in range(1, 10):
            print(f"\n── Soggetto {subject_id} ──")
            
            hist = SubjectHistory(subject_id)

            train_ds = BCIIVDataset(
                data_dir=DATA_DIR, subjects=[subject_id],
                labels=LABELS_BCICIV, mode="train",
                window_sec=self.win_sec, sfreq=SAMPLING_FREQ_BCICIV, tmin=0.0,
            )
            test_ds = BCIIVDataset(
                data_dir=DATA_DIR, subjects=[subject_id],
                labels=LABELS_BCICIV, mode="test",
                window_sec=self.win_sec, sfreq=SAMPLING_FREQ_BCICIV, tmin=0.0,
            )

            train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

            # modello fresco per ogni soggetto
            model = MultiBranchCNNLSTM(
                n_classes=N_CLASSES_BCICIV,
                win_samples=WIN_SAMPLES,
                n_channels=N_CHANNELS_BCICIV,
            ).to(DEVICE)

            crit   = nn.CrossEntropyLoss()
            optim_ = optim.AdamW(model.parameters(),
                                 lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

            best_acc, patience_cnt = 0.0, 0

            for ep in range(1, MAX_EPOCHS + 1):
                tl, ta = run_epoch(model, train_ld, optim_, crit, train=True)
                vl, va = run_epoch(model, test_ld,  optim_, crit, train=False)

                hist.update(tl, ta, vl, va)

                improved      = va > best_acc + 1e-4
                best_acc      = max(best_acc, va)
                patience_cnt  = 0 if improved else patience_cnt + 1

                print(f"  Ep {ep:02d}/{MAX_EPOCHS} | "
                      f"TrL {tl:.4f} TrA {ta:.3f} | "
                      f"VaL {vl:.4f} VaA {va:.3f} | "
                      f"Best {best_acc:.3f} | "
                      f"{'↑' if improved else ' '} pat {patience_cnt}/{PATIENCE}")

                if patience_cnt >= PATIENCE:
                    print("  Early-stop.")
                    break

            histories.append(hist)
            self.results.append(hist.best_acc)

        mean, std = np.mean(self.results), np.std(self.results)
        print("\n" + "-" * 60)
        print(f"{self.name} | Per soggetto: {[f'{r:.3f}' for r in self.results]}")
        print(f"Media {mean:.3f} ± {std:.3f}")
        print("-" * 60)


# ================================================================
# MAIN
# ================================================================
if __name__ == "__main__":
    exp = Experiment("MultiBranchCNNLSTM-BCICIV2a")
    exp.run()


 MultiBranchCNNLSTM-BCICIV2a | WIN=0.5s | BCI IV 2a

── Soggetto 1 ──


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


[DEBUG] Files ['A01T.gdf'] | Macro 288 | Sub 2304 | per class {'LEFT': 72, 'RIGHT': 72, 'FOOT': 72, 'TONGUE': 72}


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


[DEBUG] Files ['A01E.gdf'] | Macro 288 | Sub 2304 | per class {'LEFT': 72, 'RIGHT': 72, 'FOOT': 72, 'TONGUE': 72}
  Ep 01/50 | TrL 1.4973 TrA 0.264 | VaL 1.3843 VaA 0.281 | Best 0.281 | ↑ pat 0/50
  Ep 02/50 | TrL 1.3055 TrA 0.514 | VaL 1.3758 VaA 0.333 | Best 0.333 | ↑ pat 0/50
  Ep 03/50 | TrL 1.1721 TrA 0.681 | VaL 1.3700 VaA 0.344 | Best 0.344 | ↑ pat 0/50
  Ep 04/50 | TrL 1.0561 TrA 0.809 | VaL 1.3773 VaA 0.333 | Best 0.344 |   pat 1/50
  Ep 05/50 | TrL 0.9466 TrA 0.865 | VaL 1.3818 VaA 0.354 | Best 0.354 | ↑ pat 0/50
  Ep 06/50 | TrL 0.8508 TrA 0.934 | VaL 1.3955 VaA 0.392 | Best 0.392 | ↑ pat 0/50
  Ep 07/50 | TrL 0.7576 TrA 0.948 | VaL 1.3751 VaA 0.375 | Best 0.392 |   pat 1/50
  Ep 08/50 | TrL 0.6342 TrA 1.000 | VaL 1.3717 VaA 0.399 | Best 0.399 | ↑ pat 0/50
  Ep 09/50 | TrL 0.5634 TrA 0.997 | VaL 1.3724 VaA 0.406 | Best 0.406 | ↑ pat 0/50
  Ep 10/50 | TrL 0.4981 TrA 0.990 | VaL 1.3622 VaA 0.392 | Best 0.406 |   pat 1/50
  Ep 11/50 | TrL 0.4234 TrA 1.000 | VaL 1.3804 VaA 0.431

...

[DEBUG] Files ['A08E.gdf'] | Macro 288 | Sub 2304 | per class {'LEFT': 72, 'RIGHT': 72, 'FOOT': 72, 'TONGUE': 72}
  Ep 01/50 | TrL 1.5235 TrA 0.236 | VaL 1.3841 VaA 0.295 | Best 0.295 | ↑ pat 0/50
  Ep 02/50 | TrL 1.2971 TrA 0.451 | VaL 1.3749 VaA 0.281 | Best 0.295 |   pat 1/50
  Ep 03/50 | TrL 1.1525 TrA 0.622 | VaL 1.3788 VaA 0.281 | Best 0.295 |   pat 2/50
  Ep 04/50 | TrL 1.0300 TrA 0.781 | VaL 1.3802 VaA 0.306 | Best 0.306 | ↑ pat 0/50
  Ep 05/50 | TrL 0.9375 TrA 0.858 | VaL 1.4002 VaA 0.312 | Best 0.312 | ↑ pat 0/50
  Ep 06/50 | TrL 0.8175 TrA 0.944 | VaL 1.3990 VaA 0.326 | Best 0.326 | ↑ pat 0/50
  Ep 07/50 | TrL 0.7172 TrA 0.976 | VaL 1.4006 VaA 0.323 | Best 0.326 |   pat 1/50
  Ep 08/50 | TrL 0.6389 TrA 0.979 | VaL 1.4221 VaA 0.340 | Best 0.340 | ↑ pat 0/50
  Ep 09/50 | TrL 0.5461 TrA 0.997 | VaL 1.4056 VaA 0.337 | Best 0.340 |   pat 1/50
  Ep 10/50 | TrL 0.4771 TrA 0.997 | VaL 1.4186 VaA 0.337 | Best 0.340 |   pat 2/50
  Ep 11/50 | TrL 0.4321 TrA 0.997 | VaL 1.4169 VaA 0.354

/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


[DEBUG] Files ['A09T.gdf'] | Macro 288 | Sub 2304 | per class {'LEFT': 72, 'RIGHT': 72, 'FOOT': 72, 'TONGUE': 72}


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


[DEBUG] Files ['A09E.gdf'] | Macro 288 | Sub 2304 | per class {'LEFT': 72, 'RIGHT': 72, 'FOOT': 72, 'TONGUE': 72}
  Ep 01/50 | TrL 1.4732 TrA 0.281 | VaL 1.3834 VaA 0.243 | Best 0.243 | ↑ pat 0/50
  Ep 02/50 | TrL 1.2697 TrA 0.542 | VaL 1.3749 VaA 0.319 | Best 0.319 | ↑ pat 0/50
  Ep 03/50 | TrL 1.1397 TrA 0.701 | VaL 1.3646 VaA 0.319 | Best 0.319 |   pat 1/50
  Ep 04/50 | TrL 1.0218 TrA 0.833 | VaL 1.3569 VaA 0.375 | Best 0.375 | ↑ pat 0/50
  Ep 05/50 | TrL 0.9213 TrA 0.896 | VaL 1.3710 VaA 0.410 | Best 0.410 | ↑ pat 0/50
  Ep 06/50 | TrL 0.8062 TrA 0.938 | VaL 1.3758 VaA 0.382 | Best 0.410 |   pat 1/50
  Ep 07/50 | TrL 0.7222 TrA 0.983 | VaL 1.3556 VaA 0.406 | Best 0.410 |   pat 2/50
  Ep 08/50 | TrL 0.6484 TrA 0.983 | VaL 1.3542 VaA 0.389 | Best 0.410 |   pat 3/50
  Ep 09/50 | TrL 0.5376 TrA 0.993 | VaL 1.4086 VaA 0.406 | Best 0.410 |   pat 4/50
  Ep 10/50 | TrL 0.4744 TrA 0.993 | VaL 1.3710 VaA 0.406 | Best 0.410 |   pat 5/50
  Ep 11/50 | TrL 0.4157 TrA 1.000 | VaL 1.3833 VaA 0.431

In [26]:
plot_all_subjects(histories)     # curve per soggetto
plot_final_summary(histories)    # barchart riassuntivo

NameError: name 'histories' is not defined

In [24]:
raw = mne.io.read_raw_gdf(f"{DATA_DIR}/A01T.gdf", preload=True, verbose=False)
_, event_id = mne.events_from_annotations(raw, verbose=False)
print(event_id)

/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
